# RAG Indexing Pipeline

Build a ChromaDB vector store from legal, forms, and examples chunk files.

**Requirements:** Run on Kaggle with GPU T4 enabled.  
**Dataset:** `takasugi1303/rag-chunks` attached as input.

---

In [112]:
# Uncomment on first run to install dependencies
# !pip install -q chromadb sentence-transformers

In [113]:
import os
import re
import json
import time
import stat
import shutil
import hashlib
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import torch
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

print("Imports OK.")

Imports OK.


In [114]:
# ============================================================
# CONFIG  --  edit only this cell
# ============================================================

DATASET_NAME     = "rag-chunks"                                      # Kaggle dataset slug
CHUNK_DIR        = Path(f"/kaggle/input/datasets/takasugi1303/{DATASET_NAME}")
CHROMA_DIR       = Path("/kaggle/working/chromadb")
CACHE_MODEL_PATH = Path("/kaggle/working/models/embedding")
OUTPUT_ZIP       = Path("/kaggle/working/chromadb_output.zip")

CHUNK_PATHS = {
    "legal"   : CHUNK_DIR / "legal_chunks.parquet",
    "forms"   : CHUNK_DIR / "forms_chunks.parquet",
    "examples": CHUNK_DIR / "examples_chunks.parquet",
}

COLLECTION_NAMES = {
    "legal"   : "legal_chunks",
    "forms"   : "forms_chunks",
    "examples": "examples_chunks",
}

EMBED_MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
EMBED_BATCH_SIZE = 128      # T4 handles 128; lower to 64 if OOM
EMBED_DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# ── Sanity checks ──────────────────────────────────────────
print(f"Device : {EMBED_DEVICE}")
if EMBED_DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU T4.")

for name, path in CHUNK_PATHS.items():
    status = "OK" if path.exists() else "NOT FOUND"
    print(f"  [{status}] {name}: {path}")

Device : cuda
GPU    : Tesla T4
  [OK] legal: /kaggle/input/datasets/takasugi1303/rag-chunks/legal_chunks.parquet
  [OK] forms: /kaggle/input/datasets/takasugi1303/rag-chunks/forms_chunks.parquet
  [OK] examples: /kaggle/input/datasets/takasugi1303/rag-chunks/examples_chunks.parquet


In [115]:
# ============================================================
# CHROMADB CLIENT  --  always run this after any kernel restart
# ============================================================

RESET_DB = False   # <-- set False to resume partial indexing

import gc, stat, sys

# Kill only the known collection handles (don't touch Path variables)
for _name in ("chroma_client", "legal_col", "forms_col", "examples_col"):
    if _name in globals():
        globals()[_name] = None

# Unload chromadb modules so the Rust backend re-initialises from scratch
for _k in [k for k in sys.modules if k.startswith("chromadb")]:
    sys.modules.pop(_k, None)

# Re-import fresh
import chromadb
from chromadb.config import Settings

def _fix_permissions(path: Path) -> None:
    for p in path.rglob("*"):
        try:
            p.chmod(p.stat().st_mode | stat.S_IRUSR | stat.S_IWUSR)
        except OSError:
            pass

if RESET_DB and CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
    print(f"Removed stale DB at {CHROMA_DIR}")
elif CHROMA_DIR.exists():
    _fix_permissions(CHROMA_DIR)
    print(f"Fixed permissions on existing DB at {CHROMA_DIR}")

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_MODEL_PATH.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)
print(f"ChromaDB client ready: {CHROMA_DIR}")


Fixed permissions on existing DB at /kaggle/working/chromadb
ChromaDB client ready: /kaggle/working/chromadb


In [116]:
# ============================================================
# EMBEDDING MODEL
# multilingual-e5-large-instruct uses asymmetric retrieval:
#   Passage (index) prefix : "passage: "
#   Query   (search) format: "Instruct: <task>\nQuery: <text>"
# Ref: https://huggingface.co/intfloat/multilingual-e5-large-instruct
# ============================================================

INSTRUCTIONS: Dict[str, str] = {
    "legal"   : "Instruct: Tim dieu khoan phap luat Viet Nam lien quan.\nQuery: ",
    "forms"   : "Instruct: Tim mau bieu hanh chinh phu hop voi nhu cau soan thao.\nQuery: ",
    "examples": "Instruct: Tim vi du van ban hanh chinh tuong tu tinh huong sau.\nQuery: ",
}

print(f"Loading model: {EMBED_MODEL_NAME} ...")
print("(First run downloads ~1.12 GB — takes 2-3 min)")
t0 = time.time()

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=EMBED_DEVICE,
    cache_folder=str(CACHE_MODEL_PATH),
)
EMBED_DIM = embed_model.get_sentence_embedding_dimension()

print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"  Embedding dim : {EMBED_DIM}  |  Max seq: {embed_model.max_seq_length}")

# ── Quick smoke test ────────────────────────────────────────
_test = embed_model.encode(["passage: Test sentence."], normalize_embeddings=True, convert_to_numpy=True)
assert _test.shape == (1, EMBED_DIM), f"Unexpected shape: {_test.shape}"
print(f"Embedding smoke test OK — shape: {_test.shape}")

Loading model: intfloat/multilingual-e5-large-instruct ...
(First run downloads ~1.12 GB — takes 2-3 min)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded in 5.2s
  Embedding dim : 1024  |  Max seq: 512
Embedding smoke test OK — shape: (1, 1024)


In [117]:
# ============================================================
# EMBEDDING HELPERS
# ============================================================

def embed_passages(
    texts: List[str],
    batch_size: int = EMBED_BATCH_SIZE,
    max_retries: int = 3,
) -> np.ndarray:
    """Embed a list of passage strings for indexing (prefix 'passage: ').

    Automatically halves batch_size on GPU OOM and retries.
    """
    prefixed = [f"passage: {t}" for t in texts]
    for attempt in range(1, max_retries + 1):
        try:
            return embed_model.encode(
                prefixed,
                batch_size=batch_size,
                show_progress_bar=False,
                normalize_embeddings=True,
                convert_to_numpy=True,
            )
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower() and attempt < max_retries:
                batch_size = max(16, batch_size // 2)
                print(f"  OOM — retrying with batch_size={batch_size} (attempt {attempt})")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            else:
                raise
    raise RuntimeError("embed_passages: exceeded max retries")


def embed_query(query: str, collection: str = "legal") -> np.ndarray:
    """Embed a single query string for retrieval."""
    instruction = INSTRUCTIONS.get(collection, INSTRUCTIONS["legal"])
    return embed_model.encode(
        f"{instruction}{query}",
        normalize_embeddings=True,
        convert_to_numpy=True,
    )


# ── Smoke test ──────────────────────────────────────────────
_pv = embed_passages(["Dieu 1. Pham vi dieu chinh."])
_qv = embed_query("quy dinh ve hoa giai o co so", collection="legal")
print(f"embed_passages OK — shape: {_pv.shape}")
print(f"embed_query    OK — shape: {_qv.shape}")


embed_passages OK — shape: (1, 1024)
embed_query    OK — shape: (1024,)


In [118]:
# ============================================================
# CHROMADB COLLECTION HELPERS
# ============================================================

def get_or_create_collection(
    name: str,
    reset: bool = False,
) -> chromadb.Collection:
    """Return (or create) a ChromaDB collection.

    If reset=True the existing collection is deleted first.
    Uses cosine distance to match the L2-normalised embeddings.
    """
    # FIX: use the module-level chroma_client, not a new instance.
    # Creating a second PersistentClient against the same path
    # causes the sqlite readonly error.
    if reset:
        try:
            chroma_client.delete_collection(name)
            print(f"  Deleted collection: {name}")
        except Exception:
            pass  # didn't exist yet

    collection = chroma_client.get_or_create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"},   # cosine fits normalised embeddings
    )
    return collection


def _make_chunk_id(row: pd.Series) -> str:
    """Stable, collision-resistant ID derived from row content."""
    # FIX: use a deterministic hash rather than relying on the DataFrame
    # index, which can change across runs / after filtering.
    raw = f"{row.get('doc_id','')}|{row.get('chunk_index','')}|{str(row.get('text',''))[:200]}"
    return hashlib.sha256(raw.encode()).hexdigest()[:24]


def _coerce_metadata(meta: Dict) -> Dict:
    """Ensure all metadata values are ChromaDB-compatible (str/int/float/bool).

    FIX: ChromaDB rejects None, lists, dicts, and numpy scalars.
    """
    clean: Dict = {}
    for k, v in meta.items():
        if v is None:
            continue                        # skip nulls entirely
        if isinstance(v, (np.integer,)):
            clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            clean[k] = float(v)
        elif isinstance(v, (np.bool_,)):
            clean[k] = bool(v)
        elif isinstance(v, (list, dict)):
            clean[k] = json.dumps(v, ensure_ascii=False)  # serialise complex types
        elif isinstance(v, str):
            clean[k] = v
        elif isinstance(v, (int, float, bool)):
            clean[k] = v
        else:
            clean[k] = str(v)             # fallback: stringify
    return clean


def build_metadata(row: pd.Series, meta_cols: List[str]) -> Dict:
    """Build a clean metadata dict for one chunk row."""
    raw = {col: row.get(col) for col in meta_cols if col in row.index}
    return _coerce_metadata(raw)


print("Collection helpers defined.")

Collection helpers defined.


In [125]:
# ============================================================
# MAIN INDEXING FUNCTION
# ============================================================

def index_chunks(
    df: pd.DataFrame,
    collection: chromadb.Collection,
    text_col: str = "text",
    meta_cols: Optional[List[str]] = None,
    upsert_batch: int = 512,
    embed_batch: int = EMBED_BATCH_SIZE,
) -> int:
    """Embed and upsert all rows in df into collection.
    Returns the number of chunks upserted.
    """
    if meta_cols is None:
        meta_cols = []

    if text_col not in df.columns:
        raise ValueError(f"DataFrame has no column '{text_col}'")
    df = df[df[text_col].notna() & (df[text_col].str.strip() != "")].reset_index(drop=True)
    if df.empty:
        print("  WARNING: no valid rows to index.")
        return 0

    n_batches = (len(df) + upsert_batch - 1) // upsert_batch
    total_upserted = 0

    with tqdm(total=n_batches, desc=f"Indexing {collection.name}", unit="batch") as pbar:
        for batch_start in range(0, len(df), upsert_batch):
            batch      = df.iloc[batch_start : batch_start + upsert_batch]
            texts      = batch[text_col].tolist()
            ids        = [_make_chunk_id(row) for _, row in batch.iterrows()]
            metadatas  = [build_metadata(row, meta_cols) for _, row in batch.iterrows()]
            embeddings = embed_passages(texts, batch_size=embed_batch)

            collection.upsert(
                ids=ids,
                embeddings=embeddings.tolist(),
                documents=texts,
                metadatas=metadatas,
            )

            total_upserted += len(batch)
            pbar.update(1)
            pbar.set_postfix(chunks=f"{total_upserted:,}/{len(df):,}")

    return total_upserted

print("index_chunks defined.")

index_chunks defined.


In [126]:
# ============================================================
# INDEX: legal_chunks
# ============================================================
print("=" * 60)
print("INDEXING: legal_chunks")
print("=" * 60)

LEGAL_META_COLS = [
    "doc_id", "chunk_id_raw", "source_doc_no", "doc_name",
    "type_normalized", "article",
    "split_type", "chunk_index", "total_chunks", "word_count",
]

legal_df = pd.read_parquet(CHUNK_PATHS["legal"], engine="pyarrow")
print(f"Loaded: {len(legal_df):,} rows | columns: {list(legal_df.columns)}")

legal_col = get_or_create_collection(COLLECTION_NAMES["legal"], reset=RESET_DB)
print(f"  Collection '{COLLECTION_NAMES['legal']}': {legal_col.count():,} chunks currently")

t0 = time.time()
n  = index_chunks(legal_df, legal_col, meta_cols=LEGAL_META_COLS)
print(f"\nLegal indexing done! ({n:,} chunks upserted | {time.time()-t0:.0f}s)")
print(f"  DB total: {legal_col.count():,} chunks")

INDEXING: legal_chunks
Loaded: 593,430 rows | columns: ['chunk_id', 'doc_id', 'source_doc_no', 'ministry', 'type_normalized', 'doc_name', 'chapter_id', 'chapter_name', 'article', 'chunk_index', 'total_chunks', 'split_type', 'text', 'word_count']
  Collection 'legal_chunks': 0 chunks currently


Indexing legal_chunks:   0%|          | 0/1160 [00:00<?, ?batch/s]


Legal indexing done! (593,430 chunks upserted | 12443s)
  DB total: 593,430 chunks


In [127]:
# ============================================================
# INDEX: forms_chunks
# ============================================================
print("=" * 60)
print("INDEXING: forms_chunks")
print("=" * 60)

FORMS_META_COLS = [
    "doc_id", "form_id", "form_type", "purpose",
    "required_fields",
    "split_type", "chunk_index", "total_chunks", "word_count",
]

forms_df = pd.read_parquet(CHUNK_PATHS["forms"], engine="pyarrow")
print(f"Loaded: {len(forms_df):,} rows | columns: {list(forms_df.columns)}")

forms_col = get_or_create_collection(COLLECTION_NAMES["forms"], reset=RESET_DB)
print(f"  Collection '{COLLECTION_NAMES['forms']}': {forms_col.count():,} chunks currently")

t0 = time.time()
n  = index_chunks(forms_df, forms_col, meta_cols=FORMS_META_COLS)
print(f"\nForms indexing done! ({n:,} chunks upserted | {time.time()-t0:.1f}s)")
print(f"  DB total: {forms_col.count():,} chunks")

INDEXING: forms_chunks
Loaded: 10 rows | columns: ['chunk_id', 'doc_id', 'form_id', 'form_type', 'purpose', 'required_fields', 'chunk_index', 'total_chunks', 'split_type', 'text', 'word_count']
  Collection 'forms_chunks': 0 chunks currently


Indexing forms_chunks:   0%|          | 0/1 [00:00<?, ?batch/s]


Forms indexing done! (10 chunks upserted | 0.3s)
  DB total: 10 chunks


In [128]:
# ============================================================
# INDEX: examples_chunks
# ============================================================
print("=" * 60)
print("INDEXING: examples_chunks")
print("=" * 60)

EXAMPLES_META_COLS = [
    "doc_id", "example_id", "form_id", "form_type",
    "scenario", "fields_json",
    "split_type", "chunk_index", "total_chunks", "word_count",
]

examples_df = pd.read_parquet(CHUNK_PATHS["examples"], engine="pyarrow")
print(f"Loaded: {len(examples_df):,} rows | columns: {list(examples_df.columns)}")

examples_col = get_or_create_collection(COLLECTION_NAMES["examples"], reset=RESET_DB)
print(f"  Collection '{COLLECTION_NAMES['examples']}': {examples_col.count():,} chunks currently")

t0 = time.time()
n  = index_chunks(examples_df, examples_col, meta_cols=EXAMPLES_META_COLS)
print(f"\nExamples indexing done! ({n:,} chunks upserted | {time.time()-t0:.1f}s)")
print(f"  DB total: {examples_col.count():,} chunks")

INDEXING: examples_chunks
Loaded: 150 rows | columns: ['chunk_id', 'doc_id', 'example_id', 'form_id', 'form_type', 'scenario', 'fields_json', 'chunk_index', 'total_chunks', 'split_type', 'text', 'word_count']
  Collection 'examples_chunks': 0 chunks currently


Indexing examples_chunks:   0%|          | 0/1 [00:00<?, ?batch/s]


Examples indexing done! (150 chunks upserted | 3.8s)
  DB total: 150 chunks


In [129]:
# ============================================================
# INDEXING REPORT
# ============================================================
print("=" * 60)
print("        INDEXING REPORT")
print("=" * 60)

total = 0
for key, col_name in COLLECTION_NAMES.items():
    col = chroma_client.get_collection(col_name)
    cnt = col.count()
    total += cnt
    print(f"  {col_name:<28}: {cnt:>8,} chunks")

print(f"  {'─'*42}")
print(f"  {'TOTAL':<28}: {total:>8,} chunks")

db_size_mb = sum(
    f.stat().st_size for f in CHROMA_DIR.rglob("*") if f.is_file()
) / 1024 / 1024
print(f"\n  ChromaDB path : {CHROMA_DIR}")
print(f"  Size          : {db_size_mb:.1f} MB")

        INDEXING REPORT
  legal_chunks                :  593,430 chunks
  forms_chunks                :       10 chunks
  examples_chunks             :      150 chunks
  ──────────────────────────────────────────
  TOTAL                       :  593,590 chunks

  ChromaDB path : /kaggle/working/chromadb
  Size          : 9423.8 MB


In [130]:
# ============================================================
# RETRIEVAL HELPERS
# ============================================================

def retrieve_legal(
    query: str,
    top_k: int = 5,
    where: Optional[Dict] = None,
) -> List[Dict]:
    """Retrieve legal clauses relevant to query.

    where filter examples:
      {'type_normalized': 'NGHI DINH'}
      {'source_doc_no': '30/2020/ND-CP'}
    """
    q_vec = embed_query(query, collection="legal")
    results = legal_col.query(
        query_embeddings=[q_vec.tolist()],
        n_results=top_k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "chunk_id_raw" : m.get("chunk_id_raw", ""),
            "source_doc_no": m.get("source_doc_no", ""),
            "doc_name"     : m.get("doc_name", ""),
            "article"      : m.get("article", ""),
            "type"         : m.get("type_normalized", ""),
            "distance"     : round(d, 4),
            "text"         : doc,
        }
        for doc, m, d in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


def retrieve_forms(
    query: str,
    top_k: int = 3,
    where: Optional[Dict] = None,
) -> List[Dict]:
    """Retrieve administrative form templates relevant to query."""
    q_vec = embed_query(query, collection="forms")
    results = forms_col.query(
        query_embeddings=[q_vec.tolist()],
        n_results=top_k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "chunk_id_raw": m.get("chunk_id_raw", ""),
            "form_id"     : m.get("form_id", ""),
            "form_type"   : m.get("form_type", ""),
            "purpose"     : m.get("purpose", ""),
            "distance"    : round(d, 4),
            "text"        : doc,
        }
        for doc, m, d in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


def retrieve_examples(
    query: str,
    top_k: int = 3,
    form_id: Optional[str] = None,
) -> List[Dict]:
    """Retrieve few-shot administrative document examples.

    Optionally filter to a specific form_id.
    """
    q_vec = embed_query(query, collection="examples")
    where = {"form_id": form_id} if form_id else None
    results = examples_col.query(
        query_embeddings=[q_vec.tolist()],
        n_results=top_k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "chunk_id_raw": m.get("chunk_id_raw", ""),
            "form_id"     : m.get("form_id", ""),
            "scenario"    : m.get("scenario", ""),
            "distance"    : round(d, 4),
            "text"        : doc,
        }
        for doc, m, d in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


print("Retrieval helpers defined.")

Retrieval helpers defined.


In [133]:
# ============================================================
# RETRIEVAL SMOKE TEST (CLEAN VERSION)
# ============================================================

def print_section(title: str):
    print("\n" + "=" * 60)
    print(f"{title}")
    print("=" * 60)


def test_legal_retrieval(queries, top_k=3):
    print_section("LEGAL RETRIEVAL")

    for q in queries:
        print(f"\n🔎 Query: {q}")
        results = retrieve_legal(q, top_k=top_k)

        if not results:
            print("  ⚠️ No results found")
            continue

        for i, r in enumerate(results, 1):
            print(f"\n  [{i}] dist={r.get('distance'):.4f} | {r.get('source_doc_no')}")
            print(f"      📌 {r.get('article')}")
            print(f"      📝 {r.get('text', '')[:200]}...")


def test_form_retrieval(query, top_k=3):
    print_section("FORMS RETRIEVAL")

    results = retrieve_forms(query, top_k=top_k)

    if not results:
        print("  ⚠️ No forms found")
        return

    for i, r in enumerate(results, 1):
        print(f"  [{i}] {r.get('form_id')} | dist={r.get('distance'):.4f}")
        print(f"      📄 {r.get('form_type')}")


def test_example_retrieval(query, top_k=3):
    print_section("EXAMPLES RETRIEVAL")

    results = retrieve_examples(query, top_k=top_k)

    if not results:
        print("  ⚠️ No examples found")
        return

    for i, r in enumerate(results, 1):
        print(f"\n  [{i}] {r.get('form_id')} | dist={r.get('distance'):.4f}")
        print(f"      🎯 {r.get('scenario', '')[:100]}")
        print(f"      📝 {r.get('text', '')[:200]}...")


# ============================================================
# RUN TEST
# ============================================================

test_queries = [
    "quy dinh ve hoa giai o co so",
    "dieu kien de duoc cap giay phep kinh doanh",
    "trach nhiem cua Bo truong trong ban hanh van ban",
]

test_legal_retrieval(test_queries, top_k=3)
test_form_retrieval("mau don xin viec lam", top_k=3)
test_example_retrieval("don xin nghi phep", top_k=3)


LEGAL RETRIEVAL

🔎 Query: quy dinh ve hoa giai o co so

  [1] dist=0.1119 | 04/2011/TT-BVHTTDL
      📌 Điều 1. Phạm vi điều chỉnh và đối tượng áp dụng
      📝 Điều 1. Phạm vi điều chỉnh và đối tượng áp dụng
1. Thông tư này quy định về việc thực hiện nếp sống văn minh trong việc cưới, việc tang và lễ hội được tổ chức trong phạm vi cả nước. 2. Thông tư này áp...

  [2] dist=0.1135 | 18/2022/NĐ-CP
      📌 Điều 22. Lễ đặt vòng hoa tại Đài tưởng niệm các anh hùng liệt sỹ, Lễ đặt vòng hoa và vào Lăng viếng Chủ tịch Hồ Chí Minh
      📝 Điều 22. Lễ đặt vòng hoa tại Đài tưởng niệm các anh hùng liệt sỹ, Lễ đặt vòng hoa và vào Lăng viếng Chủ tịch Hồ Chí Minh
1. Lễ đặt vòng hoa tại Đài tưởng niệm và Lễ đặt vòng hoa, vào Lăng viếng Chủ tị...

  [3] dist=0.1140 | 04/2011/TT-BVHTTDL
      📌 Điều 6. Tổ chức lễ cưới tại gia đình hoặc tại địa điểm cưới
      📝 [Điều 6. Tổ chức lễ cưới tại gia đình hoặc tại địa điểm cưới]
đài tưởng niệm liệt sĩ, nghĩa trang liệt sĩ, di tích lịch sử - văn hoá; trồng cây 

In [137]:
from IPython.display import FileLink

# Thay 'chromadb_output.zip' bằng tên/đường dẫn chính xác của bạn
# Nếu OUTPUT_ZIP là đối tượng Path, bạn có thể dùng str(OUTPUT_ZIP.name)
display(FileLink('chromadb_output.zip'))

/kaggle/working/chromadb_output.zip

In [136]:
# ============================================================
# EXPORT  --  zip ChromaDB for download
# ============================================================
print("Compressing ChromaDB for download...")

# FIX: make_archive base_name must NOT include the .zip extension;
# shutil appends it automatically.  Passing OUTPUT_ZIP directly
# (which already ends with .zip) caused double-extension files.
t0 = time.time()
shutil.make_archive(
    base_name=str(OUTPUT_ZIP.with_suffix("")),   # strip .zip — shutil adds it back
    format="zip",
    root_dir=str(CHROMA_DIR.parent),
    base_dir=CHROMA_DIR.name,
)

if not OUTPUT_ZIP.exists():
    raise FileNotFoundError(f"Archive not found at {OUTPUT_ZIP}")

zip_size_mb = OUTPUT_ZIP.stat().st_size / 1024 / 1024
print(f"Done! ({time.time()-t0:.1f}s) — {OUTPUT_ZIP.name}  ({zip_size_mb:.1f} MB)")
print()
print("Download instructions:")
print("  1. Click the Output tab on the right side of the Kaggle notebook.")
print("  2. Find chromadb_output.zip → click Download.")
print("  3. Extract locally:  unzip chromadb_output.zip -d dataset/")
print("     This creates:     dataset/chromadb/")

Compressing ChromaDB for download...
Done! (486.0s) — chromadb_output.zip  (4889.1 MB)

Download instructions:
  1. Click the Output tab on the right side of the Kaggle notebook.
  2. Find chromadb_output.zip → click Download.
  3. Extract locally:  unzip chromadb_output.zip -d dataset/
     This creates:     dataset/chromadb/


In [139]:
from IPython.display import FileLink

# Thay 'chromadb_output.zip' bằng tên/đường dẫn chính xác của bạn
# Nếu OUTPUT_ZIP là đối tượng Path, bạn có thể dùng str(OUTPUT_ZIP.name)
display(FileLink('chromadb_output.zip'))

/kaggle/working/chromadb_output.zip